#Data Science Applied to Finance
#Argimiro Arratia @2026

##HW 1
Helper script for Home Work 1, Prob 2 and Prob. 3


###Prob. 2: Global asset causal network and volatility spillover
Needed Libraries and
Data retrieval from yahoo finance

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.tsa.api import VAR #for Vector Autoregression (VAR) model to find optimal lag

In [2]:
# 1. SETUP - Asset List
assets = ["SPY", "XLK", "XLF", "XLE", "GLD", "SLV", "GSG", "BTC-USD", "ETH-USD", "TLT"]

# 2. DATA DOWNLOAD (Daily)
df = yf.download(assets, start="2018-01-01", end="2026-01-01")['Close'].dropna()
print('\n',len(df))
print(df.head())

/tmp/ipykernel_4043/1388369492.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(assets, start="2018-01-01", end="2026-01-01")['Close'].dropna()
[*********************100%***********************]  10 of 10 completed


 2011
Ticker           BTC-USD      ETH-USD         GLD        GSG        SLV  \
Date                                                                      
2018-01-02  14982.099609   884.443970  125.150002  16.370001  16.209999   
2018-01-03  15201.000000   962.719971  124.820000  16.540001  16.170000   
2018-01-04  15599.200195   980.921997  125.459999  16.540001  16.230000   
2018-01-05  17429.500000   997.719971  125.330002  16.400000  16.219999   
2018-01-08  15170.099609  1148.530029  125.309998  16.430000  16.150000   

Ticker             SPY        TLT        XLE        XLF        XLK  
Date                                                                
2018-01-02  236.562210  99.093895  25.747259  23.884142  29.872923  
2018-01-03  238.058487  99.567657  26.132854  24.012461  30.122101  
2018-01-04  239.061783  99.551926  26.290600  24.234879  30.274376  
2018-01-05  240.654984  99.267654  26.280088  24.303312  30.592764  
2018-01-08  241.095078  99.204498  26.437830  24.2690

In [3]:
print(df.tail())
##check for missing values (if there are too many in your period of study SOMETHING must be done)
nan_summary = df.isna().sum()
nan_summary

Ticker           BTC-USD      ETH-USD         GLD        GSG        SLV  \
Date                                                                      
2025-12-24  87611.960938  2945.590576  411.929993  23.320000  65.220001   
2025-12-26  87301.429688  2925.745605  416.739990  23.219999  71.120003   
2025-12-29  87138.140625  2934.538330  398.600006  23.170000  66.010002   
2025-12-30  88430.132812  2971.416748  398.890015  23.299999  68.980003   
2025-12-31  87508.828125  2967.037598  396.309998  23.059999  64.419998   

Ticker             SPY        TLT        XLE        XLF         XLK  
Date                                                                 
2025-12-24  688.499695  86.735802  44.081982  55.444992  146.118332  
2025-12-26  688.429871  86.450066  43.913086  55.335552  146.348038  
2025-12-29  685.976562  86.775215  44.330357  55.037090  145.688858  
2025-12-30  685.138916  86.568306  44.668152  54.897804  145.229431  
2025-12-31  680.062744  85.878601  44.419773  54.48990

,0
Ticker,
BTC-USD,0
ETH-USD,0
GLD,0
GSG,0
SLV,0
SPY,0
TLT,0
XLE,0
XLF,0


###Prob. 3: Neural Networks vs. LSTM vs.  Gaussian process. Forecasting SP500 with fundamental indicators
Data is from goyal-welch2022Monthly.csv. monthly data with Date as yyyymm

In [ ]:
!pip install arch
!pip install dcor

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel, Matern, DotProduct
from sklearn.neural_network import MLPRegressor, MLPClassifier

from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn import metrics
from sklearn.metrics import accuracy_score
import dcor, warnings
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import grangercausalitytests
from tabulate import tabulate
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import roc_auc_score, roc_curve, auc, ConfusionMatrixDisplay, classification_report
from arch import arch_model

In [ ]:
##Libraries for LSTM
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

###Evaluation metrics

Use Normalize Residual Mean Square Error (NRMSE) (similar to R^2), and from this the percentage of outperforming direct sample mean (sample expected value)

In [ ]:
from sklearn.metrics import mean_squared_error
def nrmse(y_actual, y_predicted):
  return np.sqrt(mean_squared_error(y_actual, y_predicted)/np.mean(np.sum((y_actual-np.mean(y_actual))**2)))
## np.mean(y_actual.shift(1).dropna())
##Note: function mean_squared_error takes both arguments as arrays, so fails when given (array, mean (a number)), so must write this from scratch

def pcorrect(y_actual, y_predicted):
  return (1-nrmse(y_actual, y_predicted))*100

In [ ]:
# fix random seed for reproducibility
np.random.seed(7)
tf.random.set_seed(7)

###Read the DATA

In [ ]:
gwy = pd.read_csv('goyal_welch2022Monthly.csv')
gwy.head()

,yyyymm,Index,D12,E12,b/m,tbl,AAA,BAA,lty,ntis,Rfree,infl,ltr,corpr,svar,csp,CRSP_SPvw,CRSP_SPvwx
0,187101,4.44,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,187102,4.50,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,0.004967,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,187103,4.61,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,0.004525,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,187104,4.74,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,0.004252,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,187105,4.86,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,0.004643,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
##process the Date from yyyymm to Y-m and set it as index of table (time series)
gwy['Date'] = pd.to_datetime(gwy['yyyymm'].astype(str), format='%Y%m')
gwy.set_index('Date', inplace=True)
gwy.drop('yyyymm', axis=1, inplace=True)
gwy.head()

,Index,D12,E12,b/m,tbl,AAA,BAA,lty,ntis,Rfree,infl,ltr,corpr,svar,csp,CRSP_SPvw,CRSP_SPvwx
Date,,,,,,,,,,,,,,,,,
1871-01-01,4.44,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1871-02-01,4.50,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,0.004967,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1871-03-01,4.61,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,0.004525,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1871-04-01,4.74,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,0.004252,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1871-05-01,4.86,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,0.004643,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Remove commas and convert 'Index' to float
gwy['Index'] = gwy['Index'].str.replace(',', '', regex=False).astype(float)

In [ ]:
##check for missing values (if there are too many in your period of study SOMETHING must be done)
nan_summary = gwy.isna().sum()
nan_summary

,0
Index,0
D12,0
E12,0
b/m,602
tbl,588
AAA,576
BAA,576
lty,576
ntis,671
Rfree,1


In [ ]:
# Define the fundamental features
 # dividend-price ratio
gwy['dp'] = np.log(gwy['D12']) - np.log(gwy['Index'])

# dividend-payout ratio (de)
gwy['de'] = np.log(gwy['D12']) - np.log(gwy['E12'])

# earnings-to-price ratio
gwy['ep'] = np.log(gwy['E12']) - np.log(gwy['Index'])

# dividend yield using lag(Index)
gwy['dy'] = np.log(gwy['D12']) - np.log(gwy['Index'].shift(1))

# target
gwy['SP']= np.log(gwy['Index']) - np.log(gwy['Index'].shift(1))

# Default yield spread (dfy)= BAA-AAA rated corporate bond yields:
gwy['dfy'] = gwy['BAA'] -gwy['AAA']

# Book-to-market
gwy['bm'] = gwy['b/m']
##change of name. Drop old
gwy = gwy.drop(columns=['b/m'])

# from the table consider stock variance (svar),
## net equity expansion (ntis, start 1926), inflation (infl)
# Treasury Bill rates (tbl, 1920)

In [ ]:
gwy.head()

,Index,D12,E12,tbl,AAA,BAA,lty,ntis,Rfree,infl,...,csp,CRSP_SPvw,CRSP_SPvwx,dp,de,ep,dy,SP,dfy,bm
Date,,,,,,,,,,,,,,,,,,,,,
1871-01-01,4.44,0.26,0.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-2.837728,-0.430783,-2.406945,NaN,NaN,NaN,NaN
1871-02-01,4.50,0.26,0.4,NaN,NaN,NaN,NaN,NaN,0.004967,NaN,...,NaN,NaN,NaN,-2.851151,-0.430783,-2.420368,-2.837728,0.013423,NaN,NaN
1871-03-01,4.61,0.26,0.4,NaN,NaN,NaN,NaN,NaN,0.004525,NaN,...,NaN,NaN,NaN,-2.875302,-0.430783,-2.444519,-2.851151,0.024150,NaN,NaN
1871-04-01,4.74,0.26,0.4,NaN,NaN,NaN,NaN,NaN,0.004252,NaN,...,NaN,NaN,NaN,-2.903111,-0.430783,-2.472328,-2.875302,0.027809,NaN,NaN
1871-05-01,4.86,0.26,0.4,NaN,NaN,NaN,NaN,NaN,0.004643,NaN,...,NaN,NaN,NaN,-2.928112,-0.430783,-2.497329,-2.903111,0.025001,NaN,NaN


In [ ]:
gwy.tail()

,Index,D12,E12,tbl,AAA,BAA,lty,ntis,Rfree,infl,...,csp,CRSP_SPvw,CRSP_SPvwx,dp,de,ep,dy,SP,dfy,bm
Date,,,,,,,,,,,,,,,,,,,,,
2022-08-01,3955.00,64.8854,188.8067,0.0263,0.0407,0.0515,0.0290,-0.009732,0.0019,-0.000354,...,NaN,-0.040305,-0.042052,-4.110113,-1.068101,-3.042012,-4.153480,-0.043367,0.0108,0.227429
2022-09-01,3585.62,65.3183,187.0800,0.0313,0.0459,0.0569,0.0352,-0.011292,0.0019,0.002151,...,NaN,-0.091495,-0.092876,-4.005414,-1.052264,-2.953150,-4.103464,-0.098049,0.0110,0.249478
2022-10-01,3871.98,65.8531,182.3033,0.0372,0.0510,0.0626,0.0398,-0.015252,0.0023,0.004056,...,NaN,0.080248,0.079196,-4.074095,-1.018245,-3.055849,-3.997260,0.076835,0.0116,0.218935
2022-11-01,4080.11,66.3880,177.5267,0.0415,0.0490,0.0607,0.0389,-0.017011,0.0029,-0.001010,...,NaN,0.054166,0.052158,-4.118363,-0.983605,-3.134758,-4.066005,0.052358,0.0117,0.207182
2022-12-01,3839.50,66.9228,172.7500,0.0425,0.0443,0.0559,0.0362,-0.021246,0.0033,-0.003070,...,NaN,-0.058784,-0.060170,-4.049558,-0.948306,-3.101252,-4.110340,-0.060782,0.0116,0.216199
